# 02 - K-Nearest Neighbors
**Owner:** Syed Ayaan Hasan

K-Nearest Neighbors (KNN) classifies a building by finding the *k* most similar buildings in the training set (smallest Euclidean distance) and taking a majority vote of their damage grades, optionally weighting each vote by the inverse of its distance. KNN has no training phase: all work happens at prediction time, so its cost grows with the size of the training set.

**Design decisions**
- KNN is distance-based, so features must be on comparable scales. Numeric features are standardised and categorical features are one-hot encoded (`build_scaled_preprocessor`).
- `age` is capped at 100 before scaling (`clip_age`), because the raw data contains placeholder-like values up to 995 (see notebook 01).
- `geo_level_2_id` and `geo_level_3_id` are dropped for this model; `geo_level_1_id` is one-hot encoded (see notebook 01, section 5).
- The hyperparameters tuned are the number of neighbours *k* and the vote weighting (uniform or distance-weighted), selected by micro-F1 on the validation split.

For context, an earlier course project on this dataset (Stanford CS229, 2021) reported a micro-F1 of 0.649 for KNN on the competition test set, using 20 selected features and k = 7.

In [ ]:
import os, sys
from pathlib import Path

# --- Colab only: clone the repo and move into it ---
REPO = "ML-Project-Earthquake-Damage"
GITHUB_URL = "https://github.com/ayaanGH06/ML-Project-Earthquake-Damage.git"  # private repo: use a token URL in Colab, never commit it

if "google.colab" in sys.modules and not os.path.exists("src"):
    if not os.path.exists(REPO):
        !git clone {GITHUB_URL}
    %cd {REPO}
    !pip install -q -r requirements.txt

# --- Local Jupyter: notebooks run from notebooks/, so go up to the repo root ---
if Path.cwd().name == "notebooks":
    os.chdir("..")

# --- Data: the 3 CSVs must be in data/raw/ (see README for the download link) ---
DATA_DIR = Path("data/raw")
needed = ["train_values.csv", "train_labels.csv", "test_values.csv"]
missing = [f for f in needed if not (DATA_DIR / f).exists()]
if missing:
    raise FileNotFoundError(
        f"Missing {missing} in {DATA_DIR.resolve()}. "
        "Download them from the Google Drive link in the README and place them in data/raw/."
    )

from src.data import load_raw, train_val_split, build_scaled_preprocessor, build_tree_preprocessor
from src.evaluate import evaluate_model, micro_f1

X, y, X_test = load_raw(DATA_DIR)
X_train, X_val, y_train, y_val = train_val_split(X, y)
print(X_train.shape, X_val.shape)

## 1. Setup and baseline

In [ ]:
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors, KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer
from src.data import clip_age, AGE_CAP

FIG_DIR = Path("results/figures")
METRIC_DIR = Path("results/metrics")
for d in (FIG_DIR, METRIC_DIR):
    d.mkdir(parents=True, exist_ok=True)

majority_class = y_val.value_counts().idxmax()
majority_baseline = y_val.value_counts(normalize=True).max()
print(f"Majority-class baseline (always predict grade {majority_class}): micro-F1 = {majority_baseline:.4f}")

## 2. Feature matrix
The preprocessor is fitted on the training split only and then applied to the validation split.

In [ ]:
prep = make_pipeline(FunctionTransformer(clip_age), build_scaled_preprocessor(X_train))
Xs_train = prep.fit_transform(X_train)
Xs_val = prep.transform(X_val)
print("Feature matrix:", Xs_train.shape, Xs_val.shape)

## 3. Tuning k and the vote weighting
Finding neighbours is the expensive step. Instead of refitting a model for every setting, the 51 nearest training neighbours of every validation building are computed once. The prediction for any *k* up to 51 is then a vote over the first *k* of those neighbours, with either equal weights or weights proportional to 1 / distance.

In [ ]:
K_MAX = 51
ks = [1, 3, 5, 7, 9, 11, 15, 21, 31, 41, 51]

nn = NearestNeighbors(n_neighbors=K_MAX, algorithm="brute", n_jobs=-1).fit(Xs_train)
t0 = time.time()
dist, idx = nn.kneighbors(Xs_val)
print(f"Neighbour search for {len(Xs_val):,} validation buildings: {time.time() - t0:.0f} s")

In [ ]:
classes = np.sort(y_train.unique())
neighbour_labels = y_train.to_numpy()[idx]          # shape (n_val, K_MAX)

def predict_from_neighbours(k, weights="uniform"):
    labels = neighbour_labels[:, :k]
    if weights == "uniform":
        w = np.ones_like(labels, dtype=float)
    else:
        w = 1.0 / (dist[:, :k] + 1e-9)
    scores = np.stack([(w * (labels == c)).sum(axis=1) for c in classes], axis=1)
    return classes[scores.argmax(axis=1)]

rows = []
for k in ks:
    for weights in ("uniform", "distance"):
        rows.append({"k": k, "weights": weights,
                     "val_micro_f1": micro_f1(y_val, predict_from_neighbours(k, weights))})
sweep = pd.DataFrame(rows)
print(sweep.pivot(index="k", columns="weights", values="val_micro_f1").round(4))

best = sweep.loc[sweep["val_micro_f1"].idxmax()]
best_k, best_w = int(best["k"]), best["weights"]
print(f"\nBest setting: k = {best_k}, weights = {best_w} (validation micro-F1 = {best['val_micro_f1']:.4f})")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for weights, marker in (("uniform", "o"), ("distance", "s")):
    sub = sweep[sweep["weights"] == weights]
    ax.plot(sub["k"], sub["val_micro_f1"], marker=marker, label=f"{weights} weights")
ax.axhline(majority_baseline, color="grey", ls="--", lw=1, label="majority-class baseline")
ax.scatter([best_k], [best["val_micro_f1"]], s=140, facecolors="none", edgecolors="k", zorder=5,
           label=f"best (k={best_k}, {best_w})")
ax.set_xlabel("k (number of neighbours)")
ax.set_ylabel("validation micro-F1")
ax.set_title("KNN: effect of k and vote weighting")
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "02_knn_k_sweep.png", dpi=150, bbox_inches="tight")
plt.show()

Both weighting schemes improve steadily as k grows from 1 (micro-F1 0.613) and level off from about k = 31. For k of 5 or more, uniform weighting is consistently better than distance weighting (0.6802 against 0.6725 at k = 51). A plausible reason is that many buildings have identical or near-identical descriptions but different damage grades, so inverse-distance weights let a few very close neighbours dominate the vote, whereas a broad uniform vote is more stable. Beyond k = 31 the gain is below 0.001, so the exact choice among the largest values of k is not meaningful. The selected setting is k = 51 with uniform weights.

*k* and the weighting were selected on the validation split. With 52,121 validation buildings and a grid of 22 settings, the selection bias is small, but the score below should be read as slightly optimistic relative to completely unseen data.

## 4. Final model and evaluation

In [ ]:
knn_model = make_pipeline(
    FunctionTransformer(clip_age),
    build_scaled_preprocessor(X_train),
    KNeighborsClassifier(n_neighbors=best_k, weights=best_w, algorithm="brute", n_jobs=-1),
)
knn_model.fit(X_train, y_train)

t0 = time.time()
val_pred = knn_model.predict(X_val)
print(f"Prediction time for {len(X_val):,} buildings: {time.time() - t0:.0f} s\n")

knn_f1 = evaluate_model("knn", y_val, val_pred)

In [ ]:
with open(METRIC_DIR / "knn_best_params.json", "w") as f:
    json.dump({"n_neighbors": best_k, "weights": best_w,
               "val_micro_f1": float(knn_f1),
               "majority_baseline": float(majority_baseline)}, f, indent=2)

print(f"KNN (k={best_k}, weights={best_w}): validation micro-F1 = {knn_f1:.4f}")
print(f"Majority-class baseline:           {majority_baseline:.4f}")
print(f"Improvement over baseline:         {knn_f1 - majority_baseline:+.4f}")

The final model reaches a validation micro-F1 of 0.680, which is 0.111 above the majority-class baseline of 0.569, and its score matches the value from the tuning sweep (0.6802). This is higher than the 0.649 reported in the CS229 project, although the two numbers are not directly comparable (validation split here, competition test set there).

Performance is uneven across classes. Grade 2, the majority class, has a recall of 0.82, whereas grade 3 has a recall of 0.52 and grade 1 only 0.40. The confusion matrix shows that the errors go mostly towards the majority class: 8,295 of the 17,444 grade-3 buildings (48%) and 2,942 of the 5,025 grade-1 buildings (59%) are predicted as grade 2. Confusing the two extremes is rare (70 grade-1 buildings predicted as grade 3 and 120 grade-3 buildings predicted as grade 1), so the model seldom mistakes low damage for near-total destruction. Precision is similar across classes (0.60 to 0.69). Voting over many neighbours favours the majority class, which raises overall accuracy at the expense of the minority classes.

For reference, the untuned Random Forest baseline in `main.py` scores 0.7239 on the same validation split.

## 5. Notes on the method
- KNN produces no feature importances and no explicit model, so unlike the tree models it offers little insight into *why* a building is predicted to be damaged.
- Distances are computed in a 96-dimensional space (mostly one-hot columns), where all points tend to be similarly far apart. This weakens the notion of a "nearest" neighbour and is one reason KNN is not expected to match tree ensembles on this kind of mixed categorical data.
- Prediction time grows with the number of training buildings, which is why tuning reuses a single neighbour search rather than refitting for each setting.